<a href="https://colab.research.google.com/github/Apur52027/depression_detection/blob/main/all_model_if_df.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize

# Load the dataset
df = pd.read_excel('https://github.com/Apur52027/Dataset_ML/blob/main/BSMDD_v3_textcleaned%20-%2021K.xlsx?raw=true')

# 1. Remove Duplicate Texts
initial_count = len(df)
df = df.drop_duplicates(subset=['text'])
print(f"Removed {initial_count - len(df)} duplicate texts.")

# 2. Remove Short Texts (less than 30 words)
# Word count for Bengali is tricky. We'll split by spaces as a proxy.
df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))
df = df[df['word_count'] >= 30].drop(columns=['word_count'])
print(f"Removed texts with less than 30 words. Remaining rows: {len(df)}")

Removed 6 duplicate texts.
Removed texts with less than 30 words. Remaining rows: 19178


In [ ]:
# 3. Remove English characters and numbers
# This regex keeps Bengali unicode range (\u0980-\u09FF), whitespace, and a few punctuations.
def remove_english_and_numbers(text):
    # Keep Bengali chars, spaces, and basic punctuation (। ? !)
    bengali_pattern = re.compile(r'[^\u0980-\u09FF\s\?\!।]', re.UNICODE)
    return re.sub(bengali_pattern, '', str(text))

df['cleaned_text'] = df['text'].apply(remove_english_and_numbers)

# 4. Remove repeated punctuations
def remove_repeated_punct(text):
    # Replace 2 or more '?' or '!' or '।' with a single instance
    text = re.sub(r'([\?\!।])\1+', r'\1', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['cleaned_text'].apply(remove_repeated_punct)

# At this point, you have a 'cleaned_text' column ready for tokenization.

In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
import unicodedata # Added this import
# Download stopwords if not already
nltk.download('stopwords')

bangla_stopwords = set(stopwords.words('bengali'))

def preprocess_text(text):
    # Tokenize
     # Tokenization
    tokens = text.split()
    # remove stopword
    tokens = [word for word in tokens if word not in bangla_stopwords]
    process = " ".join(tokens)
    return process # Added return statement


df['final_text'] = df['cleaned_text'].apply(preprocess_text)

# Final dataset
X = df['final_text']  # Features
y = df['label']       # Labels (1 for depressed, 0 for non-depressed)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import re

def check_unwanted_characters(text):
    # This pattern finds any character that is NOT a Bengali character (\u0980-\u09FF),
    # a space (\s), or one of the allowed punctuations (\?|!|\u0964 - for Bengali full stop).
    # It also explicitly excludes English letters and numbers, which should already be gone.
    unwanted_pattern = re.compile(r'[^\u0980-\u09FF\s\?!\u0964]', re.UNICODE)
    found_unwanted = set(unwanted_pattern.findall(str(text)))
    return found_unwanted

all_unwanted_chars = set()
for text in df['final_text']:
    unwanted = check_unwanted_characters(text)
    if unwanted:
        all_unwanted_chars.update(unwanted)

if all_unwanted_chars:
    print(f"Found unwanted characters in 'final_text': {', '.join(sorted(list(all_unwanted_chars)))}")
else:
    print("No unwanted characters (non-Bengali, non-punctuation, non-space) found in 'final_text'.")

# Optional: Display some rows where unwanted characters might have been found (if any)
# For demonstration, let's just show a few rows of final_text to confirm its content
print("\nSample of 'final_text' column:")
display(df['final_text'].head())

No unwanted characters (non-Bengali, non-punctuation, non-space) found in 'final_text'.

Sample of 'final_text' column:


,final_text
0,মানসিক শারীরিকভাবে অসুস্থ ক্লান্ত পুরো জীবন শা...
1,দয়া সাথে থাকুন অত্যন্ত দীর্ঘ আপনাকে পড়তে উত্...
2,জানতাম সাথে ভুল লোক খারাপ জীবন কাটিয়েছে সম্পূ...
3,অনেটিভ ইংরেজি স্পিকারের অনুসরণ বিরক্তিকর অপ্রত...
4,অনেটিভ ইংরেজি স্পিকারের অনুসরণ বিরক্তিকর অপ্রত...


In [ ]:
!pip install fasttext

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.model_selection import train_test_split
X = df['final_text']
y = df['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
!pip install xgboost

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier # Corrected import
import lightgbm as lgb
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# pipeline = Pipeline([
#     ("tfidf", TfidfVectorizer(

#         max_features=12000,
#         min_df=3,
#         max_df=0.85,
#         ngram_range=(1, 2),
#         token_pattern=r'(?u)\b[\w]+\b',
#         sublinear_tf=True
#     )),
#     ("clf", LogisticRegression(max_iter=100,penalty="l2",random_state=42,class_weight="None")),
#     ('lgr', LogisticRegression(max_iter=200,penalty='l2')),
#     ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
#     ('xgb', XGBClassifier(random_state=42)),
#     ('lgbm', lgb.LGBMClassifier(random_state=42)),
#     ('dt', DecisionTreeClassifier(random_state=42)),
#     ('knn', KNeighborsClassifier()),
#     ('svm', SVC(random_state=42)),
#     ('nb', MultinomialNB())

# ])

In [ ]:
from sklearn.metrics import recall_score

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

# Common TF-IDF
tfidf = TfidfVectorizer(
    max_features=12000,
    min_df=3,
    max_df=0.85,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b[\w]+\b',
    sublinear_tf=True
)

# Different models
models = {
    "lr": LogisticRegression(
    max_iter=200,
    C=1.0,
    penalty='l2',
    solver='lbfgs',
    random_state=42,
    class_weight="balanced"
),
    "rf": RandomForestClassifier(
    n_estimators=300,       # বেশি tree → better stability
    max_depth=None,         # default, but explicitly দিলে clear হয়
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
),
    "svm":SVC(
    kernel='rbf',        # most commonly used
    C=1.0,               # regularization
    gamma='scale',       # auto ভালো কাজ করে
    probability=True,    # probability দরকার হলে
    random_state=42
),
    "nb": MultinomialNB(
    alpha=0.5,   # smoothing (tune করা যায়: 0.1, 0.5, 1.0)
    fit_prior=True
),
    'xgb': XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
),
    'lgbm': lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
),
    'dt': DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    random_state=42
),
    'knn': KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform',
    algorithm='auto',
    metric='minkowski',
    p=2
),
     'gd': GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
),
    'ada': AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)



}



In [ ]:
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ("tfidf", tfidf),
        ("clf", model)
    ])

# Train all
results = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_train_pred = pipe.predict(X_train)

    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "train_score": accuracy_score(y_train, y_train_pred),
        "test_score": accuracy_score(y_test, y_pred),
    }

    print(f"{name} accuracy:", results[name]["accuracy"])

lr accuracy: 0.87851929092805
rf accuracy: 0.8641814389989573
svm accuracy: 0.8813868613138686
nb accuracy: 0.867570385818561
xgb accuracy: 0.8743482794577685
[LightGBM] [Info] Number of positive: 6691, number of negative: 8651
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.032595 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 600463
[LightGBM] [Info] Number of data points in the train set: 15342, number of used features: 10754
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.436123 -> initscore=-0.256912
[LightGBM] [Info] Start training from score -0.256912


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm accuracy: 0.877215849843587
dt accuracy: 0.7807612095933264
knn accuracy: 0.7625130344108446
gd accuracy: 0.8493222106360793
ada accuracy: 0.7815432742440042


In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("test_score", ascending=False)

results_df

,accuracy,recall,train_score,test_score
svm,0.881387,0.861327,0.980641,0.881387
lr,0.878519,0.876270,0.911420,0.878519
lgbm,0.877216,0.860132,0.980576,0.877216
xgb,0.874348,0.851166,0.972755,0.874348
nb,0.867570,0.863718,0.884435,0.867570
rf,0.864181,0.825463,1.000000,0.864181
gd,0.849322,0.803945,0.871594,0.849322
ada,0.781543,0.614465,0.780081,0.781543
dt,0.780761,0.679617,0.791748,0.780761
knn,0.762513,0.594740,0.861100,0.762513


In [ ]:
results_df.style.highlight_max(axis=0)

,accuracy,recall,train_score,test_score
svm,0.881387,0.861327,0.980641,0.881387
lr,0.878519,0.876270,0.911420,0.878519
lgbm,0.877216,0.860132,0.980576,0.877216
xgb,0.874348,0.851166,0.972755,0.874348
nb,0.867570,0.863718,0.884435,0.867570
rf,0.864181,0.825463,1.000000,0.864181
gd,0.849322,0.803945,0.871594,0.849322
ada,0.781543,0.614465,0.780081,0.781543
dt,0.780761,0.679617,0.791748,0.780761
knn,0.762513,0.594740,0.861100,0.762513


In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf', 'poly'],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1]
}
svm= SVC(random_state=42)


# GridSearchCV setup
grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2
)

# Fit model
grid_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)